In [ ]:
try:
  import pdfplumber
  from sentence_transformers import SentenceTransformer,util
except:
  !pip install pdfplumber -q
  !pip install sentence_transformers
  import pdfplumber
  from sentence_transformers import SentenceTransformer,util

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 118.6 MB/s eta 0:00:00


In [ ]:
try:
  import faiss
except:
  !pip install faiss-cpu
  import faiss

import faiss
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 29.6 MB/s eta 0:00:00


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from transformers import AutoTokenizer , AutoModelForSeq2SeqLM
embed_model = SentenceTransformer('all-MiniLM-L6-v2',device=device)
embed_model.max_seq_length = 512
llm_model = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(llm_model)
model = AutoModelForSeq2SeqLM.from_pretrained(llm_model).to(device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 

In [ ]:
chunks = []
index = None

In [ ]:
chunk_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
def get_token_length(text):
    return len(chunk_tokenizer.encode(text))

def chunk_text(text, max_tokens=300, overlap_sentences=1):
    sentences = sent_tokenize(text)

    chunks = []
    current_chunk = []
    current_lengths = []
    current_total = 0

    for sentence in sentences:
        sentence_length = get_token_length(sentence)

        if current_total + sentence_length > max_tokens:
            if current_chunk:
                chunks.append(" ".join(current_chunk))

            # overlap handling
            overlap = current_chunk[-overlap_sentences:] if current_chunk else []
            overlap_lengths = current_lengths[-overlap_sentences:] if current_lengths else []

            current_chunk = overlap.copy()
            current_lengths = overlap_lengths.copy()
            current_total = sum(current_lengths)

        current_chunk.append(sentence)
        current_lengths.append(sentence_length)
        current_total += sentence_length

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [ ]:
def preprocess(file):
  text_chunks = []
  with pdfplumber.open(file.name) as pdf:
      for page_num , page in enumerate(pdf.pages):
        page_text = page.extract_text()
        if page_text and page_text.strip():
          chunks = chunk_text(page_text)
          for chunk in chunks:
            text_chunks.append({
                "text" : chunk,
                "page_number" : page_num + 1
            })
  return text_chunks

In [ ]:
def process_pdf(file):
  global chunks , index
  if file is None:
    return "Upload a PDF"
  chunks = preprocess(file)
  print(f"Number of chunks after preprocess: {len(chunks) if chunks else 0}")
  if not chunks:
    return "No text found in PDF"
  texts = [c["text"] for c in chunks]
  embeddings = embed_model.encode(texts,convert_to_numpy=True,normalize_embeddings=True).astype("float32")
  print(f"Shape of embeddings: {embeddings.shape}")
  dimension = embeddings.shape[1]
  index = faiss.IndexFlatIP(dimension)
  index.add(np.array(embeddings))
  return "PDF processed successfully"

In [ ]:
def chat_fn(query,history):
  global chunks , index
  query_embeddings = np.array(embed_model.encode([query],convert_to_numpy=True,normalize_embeddings=True)).astype("float32")
  k = 8
  if index is None:
    return "Please upload and process a PDF first"
  distances, indices = index.search(query_embeddings, k)
  threshold = 0.35
  filtered = [
    (chunks[i], d)
    for i, d in zip(indices[0], distances[0])
    if i!= -1 and d > threshold
]

# sort by similarity (highest first)
  filtered = sorted(filtered, key=lambda x: x[1], reverse=True)
  filtered_chunks = [c for c, _ in filtered]
  if not filtered_chunks:
    # fallback: use top-k results even if below threshold
    filtered_chunks = [
        chunks[i]
        for i in indices[0]
        if i != -1
    ]
  MAX_TOKENS = 512
  current_tokens = 0
  result = ""

  for c in filtered_chunks:
      chunk_tokens = len(tokenizer.encode(c["text"]))

      if current_tokens + chunk_tokens < MAX_TOKENS:
          result += c["text"] + " "
          current_tokens += chunk_tokens
      else:
          break
  sources = list(set(
    f"Page {c['page_number']}"
    for c, d in filtered
    if d > threshold
  ))
  chat_history = ""
  for q, a in history:
    chat_history += f"User: {q}\nAssistant: {a}\n"
  prompt = f"""
    You are a helpful AI assistant.

    Answer the question using the provided context when relevant.
    If the context does not fully answer the question, you may use your general knowledge.

    Make sure the answer is clear and complete.

    If information comes from the context, rely on it.
    If you use outside knowledge, ensure it is accurate.

    Conversation:
    {chat_history}

    Context:
    {result}

    Question:
    {query}

    Answer:
    """
  inputs = tokenizer(prompt,return_tensors="pt",truncation=True).to(device)
  output = model.generate(**inputs,max_new_tokens=512)
  response = tokenizer.decode(output[0],skip_special_tokens=True)
  if sources:
    return f"{response} \n Sources: {', '.join(sources)}"
  else:
    return f"{response} \n Sources : Model Knowledge"

In [ ]:
def reset():
  global chunks,index
  chunks = []
  index = None
  return "Reset successful"

In [ ]:
import gradio as gr
with gr.Blocks(title="AI PDF Chatbot") as demo:
  #1 File upload
  file_upload = gr.File(label="Upload PDF")
  process_btn = gr.Button("Process PDF")
  #Status creation
  status = gr.Textbox()
  #Processing
  process_btn.click(fn=process_pdf,inputs=file_upload,outputs=status)
  #2 Chat Interface
  chatbot = gr.ChatInterface(
      fn = chat_fn
  )
  #Reset button
  reset_btn = gr.Button("End Conversation")
  reset_btn.click(fn=reset,outputs=status)
demo.launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://dfbaf1bca0bd246a2a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Number of chunks after preprocess: 97
Shape of embeddings: (97, 384)
